# `klygo.config` (`environment.py`): Environment variable overlays

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Environment.ipynb)


Overlay prefixed environment variables (`KLYGO_*` or custom prefixes) onto base configurations with optional JSON scalar parsing.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-xlwzy9h3


## Jump to an API

[`from_env`](#from_env)

<a id="from_env"></a>

### `config.from_env`

Overlay matching environment variables onto a configuration.

**Signature:** `config.from_env(config_data: Mapping[str, Any] | box.Box | None = None, prefix: str = 'KLYGO_', sep: str = '_', parse_values: bool = False) -> dict[str, Any]`

In [2]:
import os

os.environ["DEMO_MODEL_BATCH"] = "64"
os.environ["DEMO_MODEL_AMP"] = "true"
try:
    raw_cfg = config.from_env({"model": {"name": "yolov8n"}}, prefix="DEMO_", parse_values=False)
    parsed_cfg = config.from_env({"model": {"name": "yolov8n"}}, prefix="DEMO_", parse_values=True)
    print("Raw string mode:", raw_cfg)
    print("Parsed JSON scalar mode:", parsed_cfg)
finally:
    os.environ.pop("DEMO_MODEL_BATCH", None)
    os.environ.pop("DEMO_MODEL_AMP", None)

Raw string mode: {'model': {'name': 'yolov8n', 'batch': '64', 'amp': 'true'}}
Parsed JSON scalar mode: {'model': {'name': 'yolov8n', 'batch': 64, 'amp': True}}


In [3]:
# Error example: empty separator or non-bool parse_values
for kwargs in ({'sep': ''}, {'parse_values': 'yes'}):
    try:
        config.from_env(**kwargs)
    except (ValueError, TypeError) as exc:
        print(type(exc).__name__, "->", exc)

ValueError -> sep must be a non-empty string
TypeError -> parse_values must be bool, got str


## Put it together

In [4]:
import os
os.environ["KLYGO_DEMO_TRAIN__BATCH"] = "64"
os.environ["KLYGO_DEMO_TRAIN__AMP"] = "true"
try:
    overlaid = config.from_env({"train": {"batch": 16, "amp": False}}, prefix="KLYGO_DEMO_")
    print("Overlaid from env:", overlaid["train"])
finally:
    os.environ.pop("KLYGO_DEMO_TRAIN__BATCH", None)
    os.environ.pop("KLYGO_DEMO_TRAIN__AMP", None)

Overlaid from env: {'batch': '64', 'amp': 'true'}


## Remember

- `from_env()` strips `prefix`, maps `separator` (default `'__'`) to nested keys, and automatically parses booleans, numbers, JSON literals, and `null`/`none` when `parse_values=True`.
- Next in `klygo.config`: continue with [Document_Config_Validation.ipynb](Document_Config_Validation.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).